In [2]:
#kiểm tra có có tài khoản giống nhau giữa các ngân hàng không
import pandas as pd

In [3]:
account_node=pd.read_csv("dataset_high/HI-Small_accounts.csv")

In [4]:
account_node.head()

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,Portugal Bank #4507,331579,80B779D80,80062E240,Sole Proprietorship #50438
1,Canada Bank #27,210,809D86900,800C998A0,Corporation #33520
2,UK Bank #33,21884,80812BE00,800C47F50,Partnership #35397
3,Germany Bank #4815,32742,81047F300,80096F0B0,Corporation #48813
4,National Bank of Harrisburg,127390,80BD8CF00,800FB8760,Corporation #889


In [5]:
df_clean = account_node[['Bank ID','Account Number']].drop_duplicates()

In [6]:
sum_unique=df_clean['Account Number'].nunique()

In [7]:
print(sum_unique)

518573


In [8]:
pairs=df_clean.groupby(['Account Number','Bank ID']).ngroups

In [9]:
print(pairs)

518581


In [14]:
#kiểm tra dữ liệu split theo ngày
dtype = {"From Bank": str, "To Bank": str, "Account": str, "Account.1": str}
df_day_split = pd.read_csv("dataset_high/HI-Small_trans.csv", dtype=dtype)

In [15]:
df_day_split["day"] = df_day_split["Timestamp"].str.slice(0, 10)
order = df_day_split["Timestamp"].sort_values(kind="mergesort").index
n = len(df_day_split)
days_sorted = df_day_split["day"].loc[order].to_numpy()
d1 = days_sorted[int(n * 0.6)]  
d2 = days_sorted[int(n * 0.8)]

In [17]:
import numpy as np
df_day_split["split"] = np.where(df_day_split["day"] < d1, "train",
                np.where(df_day_split["day"] < d2, "val", "test"))

print(f"Tong so giao dich: {n:,}")
print(f"Ranh gioi train|val = dau ngay {d1}")
print(f"Ranh gioi val|test  = dau ngay {d2}\n")
for s in ["train", "val", "test"]:
    sub = df_day_split[df_day_split.split == s]
    f = int((sub["Is Laundering"] == 1).sum())
    print(f"{s:5s} n={len(sub):>9,} ({len(sub)/n*100:5.2f}%)  "
          f"fraud={f:>5} rate={f/len(sub)*100:.4f}%  ngay {sub.day.min()}..{sub.day.max()}")

Tong so giao dich: 5,078,345
Ranh gioi train|val = dau ngay 2022/09/06
Ranh gioi val|test  = dau ngay 2022/09/08

train n=2,766,832 (54.48%)  fraud= 1999 rate=0.0722%  ngay 2022/09/01..2022/09/05
val   n=  964,840 (19.00%)  fraud= 1028 rate=0.1065%  ngay 2022/09/06..2022/09/07
test  n=1,346,673 (26.52%)  fraud= 2150 rate=0.1597%  ngay 2022/09/08..2022/09/18


In [18]:
#kiểm tra dữ liệu split theo index
df_index_split = pd.read_csv("dataset_high/HI-Small_trans.csv", dtype=dtype)

In [23]:
order = df_index_split["Timestamp"].sort_values(kind="mergesort").index
df_index_split=df_index_split.iloc[order].reset_index(drop=True)
n=len(df_index_split)
t1=int(n*0.6)
t2=int(n*0.8)
ts1 = df_index_split["Timestamp"].iloc[t1]
ts2 = df_index_split["Timestamp"].iloc[t2]
# gán nhãn train/test/value cho tập dữ liệu mới và xuất file
df_index_split["split"]="test"
df_index_split.loc[df_index_split["Timestamp"]<ts2,"split"]="val"
df_index_split.loc[df_index_split["Timestamp"]<ts1,"split"]="train"

In [24]:
print(f"Tong so giao dich: {n:,}")
print(f"Diem cat raw: t1=index {t1} (ts1={ts1}), t2=index {t2} (ts2={ts2})")
print(f"Ranh gioi thuc te sau tie-breaking:")
print(f"  train max Timestamp = {df_index_split[df_index_split.split=='train']['Timestamp'].max()}")
print(f"  val   min Timestamp = {df_index_split[df_index_split.split=='val']['Timestamp'].min()}")
print(f"  val   max Timestamp = {df_index_split[df_index_split.split=='val']['Timestamp'].max()}")
print(f"  test  min Timestamp = {df_index_split[df_index_split.split=='test']['Timestamp'].min()}\n")

for s in ["train", "val", "test"]:
    sub = df_index_split[df_index_split.split == s]
    f = int((sub["Is Laundering"] == 1).sum())
    print(f"{s:5s} n={len(sub):>9,} ({len(sub)/n*100:5.2f}%)  "
          f"fraud={f:>5} rate={f/len(sub)*100:.4f}%  "
          f"ngay {sub['Timestamp'].min()[:10]}..{sub['Timestamp'].max()[:10]}")

Tong so giao dich: 5,078,345
Diem cat raw: t1=index 3047007 (ts1=2022/09/06 13:36), t2=index 4062676 (ts2=2022/09/08 16:12)
Ranh gioi thuc te sau tie-breaking:
  train max Timestamp = 2022/09/06 13:35
  val   min Timestamp = 2022/09/06 13:36
  val   max Timestamp = 2022/09/08 16:11
  test  min Timestamp = 2022/09/08 16:12

train n=3,046,861 (60.00%)  fraud= 2297 rate=0.0754%  ngay 2022/09/01..2022/09/06
val   n=1,015,602 (20.00%)  fraud= 1082 rate=0.1065%  ngay 2022/09/06..2022/09/08
test  n=1,015,882 (20.00%)  fraud= 1798 rate=0.1770%  ngay 2022/09/08..2022/09/18


In [25]:
split_day=pd.read_csv("dataset_high/HI-Small_Trans_split_day.csv")

In [28]:
split_index=pd.read_csv("dataset_high/HI-Small_Trans_split_index.csv")

In [32]:
split_day["Timestamp"] = pd.to_datetime(split_day["Timestamp"])
split_index["Timestamp"] = pd.to_datetime(split_index["Timestamp"])
display(split_day[split_day["split"] == "val"][["Timestamp", "split"]].head())
display(split_index[split_index["split"] == "val"][["Timestamp", "split"]].head())

,Timestamp,split
2766832,2022-09-06,val
2766833,2022-09-06,val
2766834,2022-09-06,val
2766835,2022-09-06,val
2766836,2022-09-06,val


,Timestamp,split
3046861,2022-09-06 13:36:00,val
3046862,2022-09-06 13:36:00,val
3046863,2022-09-06 13:36:00,val
3046864,2022-09-06 13:36:00,val
3046865,2022-09-06 13:36:00,val


In [33]:
print(split_index.iloc[3046850:3046860])

                  Timestamp  From Bank    Account  To Bank  Account.1  \
3046850 2022-09-06 13:35:00     226925  81232F7F0    48308  812AC0740   
3046851 2022-09-06 13:35:00     248857  8123B48C0      222  812D8CCD0   
3046852 2022-09-06 13:35:00         15  8000E2431       15  8000E2431   
3046853 2022-09-06 13:35:00        220  813634AE1      126  813764581   
3046854 2022-09-06 13:35:00     254595  813D936B1   254565  813F5F831   
3046855 2022-09-06 13:35:00        124  813D0C341      126  81409E921   
3046856 2022-09-06 13:35:00        225  813B7DE81       15  814135F31   
3046857 2022-09-06 13:35:00     152416  813CD5FB1   154148  814340251   
3046858 2022-09-06 13:35:00        220  813760121   152627  8144127A1   
3046859 2022-09-06 13:35:00        124  813DEFDC1   254595  8144306A1   

         Amount Received Receiving Currency   Amount Paid Payment Currency  \
3046850      7677.010000        Saudi Riyal   7677.010000      Saudi Riyal   
3046851     47177.020000        Saudi Ri